In [1]:
import cv2 
import numpy as np
import pandas as pd
import torch
import torch.nn 
import json
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from scipy.spatial.distance import cosine
import torchreid
from PIL import Image
import torchvision.transforms.v2 as T

c:\ProgramData\miniconda3\Lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [2]:
class ReIDModel:
    """
    Act as a wrapper to load pretrained ReID model
    Converts player image crops into embedding vectors.
    
    Input:
        - model_name (str): name of pretrained ReID model to load, default is 'osnet_x1_0'
        - device (str): 'cude' or 'cpu'
    Output: 
        - embedding vector (np.array) feature per player representing player appearance
    """
    def __init__(self,model_name='osnet_x1_0', device='cuda'):
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        
        #Load pretrained OSNet
        self.model = torchreid.models.build_model(
            name = model_name,
            num_classes = 1000,
            pretrained = True,
        )
        self.model.eval()
        self.model = self.model.half()
        self.model.to(self.device)
        
        # Transform
        self.transform = T.Compose([
            T.Resize((256,128)),
            T.ToDtype(torch.float32, scale=True),           # normalize [0,255] → [0,1]
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])

    # Old method inefficient 
    def extract_embedding(self,image):
        """
        Extract embedding vector from player crop image
            - Converts OpenCV image (np) to PIL Image
            - Apply normalization + resize
            - Pass through ReID model
            - Return embedding vector

        Input:
            - image (np.array): cropped player image
        Output:
            - embedding vector (np.array) feature per player representing player appearance
        """
        # Handle empty crop
        if image is None or image.size ==0:
            return None
        
        #convet BGR to RGB
        image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        # Convert Numpy -> PIL
        image = Image.fromarray(image)
        
        #Apply Transforms to batch tensor
        image = self.transform(image).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            features = self.model(image)
        
        embedding = features.cpu().numpy().flatten()
        #Normalize
        embedding = embedding / np.linalg.norm(embedding)
        
        return embedding
    
    def extract_embeddings_batch(self, crops):
        """
        Extract embeddings from multiple player crops in batch
            - Converts list of NumPy images → Torch tensors
            - Applies transforms on GPU
            - Runs batch inference
            - Returns embeddings for all players

        Input:
            - crops: list of NumPy arrays (BGR images)

        Output:
            - embeddings: list of numpy vectors
        """

        valid_tensors = []

        for crop in crops:
            # Handle empty crop
            if crop is None or crop.size == 0:
                continue

            # Convert BGR → RGB
            #crop = crop[:, :, ::-1].copy()
            crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            
            # NumPy → Tensor (C,H,W)
            tensor = torch.from_numpy(crop).permute(2, 0, 1).contiguous()
            
            #Resize to ReID expected size (H = 256, W=128)
            tensor = torch.nn.functional.interpolate(
                tensor.unsqueeze(0).float(),
                size=(256,128),
                mode='bilinear',
                align_corners=False
            ).squeeze(0)
            
            valid_tensors.append(tensor)

        if len(valid_tensors) == 0:
            return []

        # Stack batch
        batch = torch.stack(valid_tensors).to(self.device)
        batch = batch.to(self.device,non_blocking=True)
        
        # Apply transforms
        batch = self.transform(batch)
         # Normalize (manual instead of torchvision overhead)
        # batch = batch / 255.0
        # mean = torch.tensor([0.485, 0.456, 0.406], device=self.device).view(1,3,1,1)
        # std = torch.tensor([0.229, 0.224, 0.225], device=self.device).view(1,3,1,1)
        # batch = (batch - mean) / std

        # Half precision
        batch = batch.half()

        with torch.no_grad():
            features = self.model(batch)

        # Convert to numpy
        embeddings = features.float().cpu().numpy()
        #Normalize
        embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

        return embeddings

In [3]:
class InputLoader:
    """
    Load tracking data from CSV or JSON into frame-based structure
    
    Input:
        - source_type: 'csv' or 'json'
        - path: file path
    Output:
        - frames_data: dict of frame to list of player detections
    """
    
    def __init__(self, source_type, path):
        self.source_type = source_type
        self.path = path

    def load(self):
        if self.source_type == 'csv':
            return self._load_csv()
        elif self.source_type == 'json':
            return self._load_json()
        
    def _load_csv(self):
        df = pd.read_csv(self.path)
        frames = defaultdict(list)
        
        for _, row in df.iterrows():
            frames[int(row['frame_id'])].append({
                "bbox": [row['x1'], row['y1'], row['x2'], row['y2']],
                "confidence": row['confidence'],
                "player_id": row.get('player_id', -1)  
            })
        return frames

    def _load_json(self):
        with open(self.path) as f:
            data = json.load(f)
        
        frames = defaultdict(list)
        for d in data:
            frames[d['frame_id']].append(d)
            print(d)
        return frames

In [4]:
class IdentityManager:
    """
    Match and mantain consistent player identities using
        - ReID embeddings similarity
        - Jersey number match
    
    Input:
        - embedding vectors per detection (np.array)
        - Jersey number (int or None)
    Output:
        - consistent track IDs
    """
    
    def __init__(self, threshold = 0.7):
        self.memory = {}
        self.next_id = 0
        self.threshold = threshold
    
    def match(self, embedding, jersey_number = None):
        if embedding is None:
            return None
        
        # ---------- 1. HARD MATCH (OCR) ----------
        if jersey_number is not None:
            for track_id, data in self.memory.items():
                if data["jersey"] == jersey_number:
                    return track_id
                
        # ---------- 2. SOFT MATCH (ReID) ----------~~
        best_id = None
        best_score = -1
        
        for track_id, data in self.memory.items():
            for e in data["embeddings"]:
                sim = 1 - cosine(embedding, e)

                if sim > best_score:
                    best_score = sim
                    best_id = track_id

        if best_score > self.threshold:
            return best_id
        
        #New ID
        new_id = self.next_id
        self.memory[new_id] = {
            "embeddings": [],
            "jersey": jersey_number
            }
        self.next_id += 1
        return new_id
    
    def update(self, track_id, embedding, jersey_number=None):
        """
        Update track memory

        Args:
            track_id (_type_): _description_
            embedding (_type_): _description_
            jersey_number (_type_): _description_
        """
        if track_id not in self.memory:
            return

        if embedding is not None:
            self.memory[track_id]["embeddings"].append(embedding)

            if len(self.memory[track_id]["embeddings"]) > 30:
                self.memory[track_id]["embeddings"].pop(0)

        # Update jersey if confident
        if jersey_number is not None:
            self.memory[track_id]["jersey"] = jersey_number

In [5]:
class VideoProcessor: 
    """
    Process video frames, apply ReID, draw bounding boxes, and produce output video and JSON results
    
    Input: 
        - video path
        - frames_data
    Output:
        - annotated video
        - tracking results list
    """
    
    def __init__(self, video_path, output_path="output.mp4"):
        self.cap = cv2.VideoCapture(video_path)
        
        width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(self.cap.get(cv2.CAP_PROP_FPS))
        
        self.writer = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps,
            (width, height)
        )
    def read_video(self):
        """
        Generator function to read video frame-by-frame

        - Reads frames from OpenCV VideoCapture
        - Yields one frame at a time
        - Stops when video ends

        Input:
            - None (uses self.cap internally)

        Output:
            - frame (np.array): video frame in BGR format
        """

        while True:
            ret, frame = self.cap.read()

            if not ret:
                break

            yield frame

        # Release resources after finishing
        self.cap.release()
        self.writer.release()
        
    def process(self, frames_data, reid_model, identity_manager, ocr_model):
        """
            Process video using batch ReID and batch OCR for effciciency
                - Extracts all crops per frame
                - Runs batch embedding extraction
                - Assign track IDs
                - Run batch oCR for jersey numbers
                - matches identities 
                - Draws tracking output

            Args:
                - frames_data (dict): A dictionary mapping frame IDs to lists of detection results.
                - reid_model (ReIDModel): The ReID model for extracting embeddings.
                - identity_manager (IdentityManager): The manager for handling identity matching.
                - ocr_model (JerseyOCR): The OCR model for extracting jersey numbers.
            Output:
                - results: list of tracking results with frame_id, track_id, bbox, confidence
        """
        results = []
        
        for frame_idx, frame in enumerate(self.read_video()):
            detections = frames_data.get(frame_idx, [])

            crops = []
            valid_dets = []

            # Collect crops
            for det in detections:
                x1, y1, x2, y2 = map(int, det['bbox'])

                crop = frame[y1:y2, x1:x2]

                if crop is None or crop.size == 0:
                    continue

                crops.append(crop)
                valid_dets.append(det)

            # Batch embeddings
            embeddings = reid_model.extract_embeddings_batch(crops)
            track_ids = []
            for det, emb in zip(valid_dets, embeddings):
                track_id = identity_manager.match(emb)
                identity_manager.update(track_id, emb)

                det['global_id'] = track_id
                det['frame_id'] = frame_idx
                track_ids.append(track_id)
            
            # Jersey ocr with temporal voting
            jersey_numbers = ocr_model.process_batch(crops, track_ids)

            # Draw box
            for det, jersey in zip(valid_dets, jersey_numbers):
                det['jersey_number'] = jersey
                results.append(det)

                x1, y1, x2, y2 = map(int, det['bbox'])
                track_id = det['global_id']

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

                label = f"ID {track_id}"
                if jersey is not None:
                    label += f" | #{jersey}"

                cv2.putText(
                    frame,
                    label,
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0,255,0),
                    2
                )

            self.writer.write(frame)

        return results

In [6]:
def save_json(results, path="output.json"):
    """
    Save tracking results to JSON file

    Input:
        results: list of tracking outputs
        path (str, optional): _description_. Defaults to "output.json".
    Output:
        - JSON file with tracking results
    """
    with open(path, "w") as f:
        json.dump(results, f, indent=4)

In [7]:
# Full pipeline implementation
from JerseyOCR.JerseyOCR import JerseyOCR

# Load tracking data (from your team)
loader = InputLoader("csv", "GoldCoast_Carlton_VFL_tracking_parsed.csv")
frames_data = loader.load()

# Initialize ReID models
reid_model = ReIDModel()
identity_manager = IdentityManager(threshold=0.7)

#Initialize OCR model
ocr_model = JerseyOCR(use_gpu=True)

# Process video
processor = VideoProcessor("GoldCoast_Carlton_VFL.mp4", "output.mp4")
results = processor.process(frames_data, reid_model, identity_manager, ocr_model)

# Save output
save_json(results, "output.json")

c:\ProgramData\miniconda3\Lib\site-packages\torchreid\reid\models\osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_file)


Successfully loaded imagenet pretrained weights from "C:\Users\PC/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"


## Building Video with Original tracking performance

In [17]:
class YOLOTrackingVisualizer:
    """
    Visualize YOLO tracking results directly from CSV (baseline model)
        - Reads video frame-by-frame
        - Draws bounding boxes from CSV tracking output
        - Displays original YOLO track IDs (NO ReID)
        - Outputs annotated video for comparison

    Input:
        - video_path (str): path to input video
        - frames_data (dict): frame_id → detections
    Output:
        - annotated video showing YOLO tracking performance
    """

    def __init__(self, video_path, output_path="yolo_baseline.mp4"):
        self.cap = cv2.VideoCapture(video_path)

        width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(self.cap.get(cv2.CAP_PROP_FPS))

        self.writer = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps,
            (width, height)
        )

    def read_video(self):
        while True:
            ret, frame = self.cap.read()
            if not ret:
                break
            yield frame

        self.cap.release()
        self.writer.release()

    def process(self, frames_data):
        results = []

        for frame_idx, frame in enumerate(self.read_video()):
            detections = frames_data.get(frame_idx, [])

            for det in detections:
                x1, y1, x2, y2 = map(int, det['bbox'])

                # YOLO ID (from CSV if exists)
                yolo_id = det.get("player_id", -1)

                # Draw RED for baseline
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,0,255), 2)

                cv2.putText(
                    frame,
                    f"YOLO ID {yolo_id}",
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0,0,255),
                    2
                )

                det['frame_id'] = frame_idx
                results.append(det)

            self.writer.write(frame)

        return results

In [18]:
# Load data
loader = InputLoader("csv", "GoldCoast_Carlton_VFL_tracking_parsed.csv")
frames_data = loader.load()

# ---------- BASELINE ----------
yolo_vis = YOLOTrackingVisualizer(
    "GoldCoast_Carlton_VFL.mp4",
    "baseline_yolo.mp4"
)

yolo_results = yolo_vis.process(frames_data)


# # ---------- REID ----------
# reid_model = ReIDModel()
# identity_manager = IdentityManager(threshold=0.7)

# processor = VideoProcessor(
#     "GoldCoast_Carlton_VFL.mp4",
#     "reid_tracking.mp4"
# )

# reid_results = processor.process(frames_data, reid_model, identity_manager)

# Adding Jersey OCR into the pipeline